# 🏦 MLOps — Tema 2: Ingeniería de Software para Ciencia de Datos
## Caso Práctico: Refactorización del Sistema de Predicción de Default Crediticio — Banco Wiesse

---

**Proyecto:** Sistema de Predicción de Default Crediticio  
**Objetivo:** Demostrar cómo el notebook monolítico fue refactorizado a un proyecto modular  
**Estructura:** Este notebook es la **versión de referencia** — el código que corre en producción vive en `src/`

> 📌 **Nota:** Este notebook ya no contiene la lógica de negocio directamente.  
> Importa los módulos refactorizados de `src/` y los ejecuta en orden.  
> Puedes ejecutar `python src/main.py` para correr el pipeline completo sin abrir este notebook.

---

### 📁 Estructura del proyecto
```
banco-wiesse-mlops/
├── data/
│   └── Dataset Endeudamiento Crediticio.csv
├── notebooks/
│   └── refactorizacion_banco_wiesse.ipynb  ← este archivo
├── src/
│   ├── __init__.py
│   ├── data_loader.py      ← Parte 1: carga + validación
│   ├── preprocessing.py    ← Parte 2-3: limpieza, imputación, balanceo
│   ├── features.py         ← Parte 2: feature engineering
│   ├── train.py            ← Parte 4: entrenamiento y evaluación
│   └── main.py             ← orquestador del pipeline
├── tests/
│   ├── test_preprocessing.py
│   └── test_features.py
└── requirements.txt
```

---
## ⚙️ Sección 0 — Configuración del entorno

Antes de importar los módulos necesitamos agregar `src/` al path de Python para que el notebook pueda encontrarlos.

In [ ]:
import sys
import os
import logging
import warnings
warnings.filterwarnings('ignore')

# Agregar src/ al path para importar los módulos refactorizados
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# Librerías para visualización (solo en el notebook)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
%matplotlib inline

# Configurar logging para ver el output de los módulos
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-20s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)

print("✅ Entorno configurado")
print(f"   Python: {sys.version.split()[0]}")
print(f"   pandas: {pd.__version__}")
print(f"   numpy:  {np.__version__}")

In [ ]:
# ── Importar todos los módulos refactorizados ──────────────────────────────
from data_loader import cargar_datos

from preprocessing import (
    limpiar_na_strings,
    winsorizar_columnas,
    imputar_nulos,
    dividir_y_balancear,
)

from features import (
    crear_score_retrasos,
    crear_categorias_edad,
    crear_categorias_dependientes,
    estandarizar_variables,
    seleccionar_features,
)

from train import (
    entrenar_modelo,
    comparar_modelos,
    analizar_errores,
    MODELOS_DEFAULT,
)

print("✅ Módulos importados correctamente")
print()
print("   📦 data_loader    → cargar_datos()")
print("   📦 preprocessing  → limpiar_na_strings(), winsorizar_columnas(),")
print("                        imputar_nulos(), dividir_y_balancear()")
print("   📦 features       → crear_score_retrasos(), crear_categorias_edad(),")
print("                        crear_categorias_dependientes(), estandarizar_variables(),")
print("                        seleccionar_features()")
print("   📦 train          → entrenar_modelo(), comparar_modelos(), analizar_errores()")

---
## 📂 Parte 1 — Carga y Entendimiento de Datos

**Módulo:** `src/data_loader.py`  
**Función:** `cargar_datos(ruta)` 

### ¿Qué hace este módulo?
- Carga el CSV con `sep=';'` y `decimal='.'` (formato original del dataset)
- Valida que existan las 11 columnas requeridas del proyecto
- Lanza `FileNotFoundError` o `ValueError` con mensajes claros si algo falla
- Registra en el log: dimensiones, distribución de `Default`

### ¿Qué había en el notebook original?
```python
# Notebook original — Celda suelta sin validación
df = pd.read_csv('Dataset Endeudamiento Crediticio.csv', sep=';', decimal='.')
print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
```

In [ ]:
# ── ANTES (notebook original) ──────────────────────────────────────────────
# df = pd.read_csv('Dataset Endeudamiento Crediticio.csv', sep=';', decimal='.')

# ── AHORA (función modular con validación) ──────────────────────────────────
RUTA_DATOS = '../data/Dataset Endeudamiento Crediticio.csv'

df = cargar_datos(RUTA_DATOS)

print()
print("=" * 60)
print("  RESUMEN DEL DATASET CARGADO")
print("=" * 60)
print(f"  Filas    : {df.shape[0]:,}")
print(f"  Columnas : {df.shape[1]}")
print(f"  Memoria  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()
print("  Columnas disponibles:")
for col in df.columns:
    print(f"    • {col} ({df[col].dtype})")

In [ ]:
# ── Insight 1: Distribución del target (Default) ───────────────────────────
default_dist    = df['Default'].value_counts()
default_percent = df['Default'].value_counts(normalize=True) * 100
ratio           = default_dist[0] / default_dist[1]

print("🎯 DISTRIBUCIÓN DEL DEFAULT (VARIABLE OBJETIVO)")
print("-" * 50)
print(f"  No Default (0) : {default_dist[0]:,} clientes  ({default_percent[0]:.1f}%)")
print(f"  Default    (1) : {default_dist[1]:,} clientes  ({default_percent[1]:.1f}%)")
print(f"  Ratio          : {ratio:.1f}:1  → Desbalanceo severo")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars = axes[0].bar(['No Default', 'Default'], default_dist.values,
                   color=['#2ecc71', '#e74c3c'], alpha=0.8)
for bar, count, perc in zip(bars, default_dist.values, default_percent.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count:,}\n({perc:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Distribución de Default', fontweight='bold')
axes[0].set_ylabel('Número de Clientes')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].pie(default_dist.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Proporción de Default', fontweight='bold')

plt.tight_layout()
plt.suptitle('Insight 1: Variable Objetivo', fontweight='bold', y=1.02)
plt.show()

In [ ]:
# ── Insight 2: Calidad de datos (valores faltantes y NA como string) ────────
print("🔍 INSIGHT 2: CALIDAD DE DATOS")
print("-" * 50)

# Detectar 'NA' como string ANTES de limpiar
for col in ['Mto_ingreso_mensual', 'Nro_dependiente']:
    na_strings = df[col].astype(str).str.upper().str.contains('NA', na=False).sum()
    na_reales  = df[col].isna().sum()
    print(f"  {col}:")
    print(f"    'NA' como string : {na_strings}")
    print(f"    NaN reales       : {na_reales}")

print()
missing = df.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    print("  Variables con nulos:")
    for col, n in missing.items():
        print(f"    • {col}: {n} ({n/len(df)*100:.1f}%)")
else:
    print("  ✓ No hay NaN visibles (los 'NA' como string aún no se han convertido)")

---
## 🔧 Parte 2 — Tratamiento de Datos

**Módulo:** `src/preprocessing.py`  
**Funciones:** `limpiar_na_strings()`, `winsorizar_columnas()`, `imputar_nulos()`

### ¿Qué hace este módulo?
1. **limpiar_na_strings:** convierte `"NA"`, `"na"`, `"N/A"` a `NaN` real y castea a numérico
2. **winsorizar_columnas:** aplica clip al percentil 5-95 para tratar outliers extremos
3. **imputar_nulos:** imputa con mediana o moda según la naturaleza de cada variable

### ¿Qué había en el notebook original?
```python
# Notebook original — Parte 2.1 (función definida en celda)
def clean_na_strings(series):
    series_clean = series.replace(['NA', 'na', 'Na', 'N/A'], np.nan)
    series_clean = pd.to_numeric(series_clean, errors='coerce')
    return series_clean

for col in ['Mto_ingreso_mensual', 'Nro_dependiente']:
    df_clean[col] = clean_na_strings(df_clean[col])
```

In [ ]:
# ── 2.1 Limpieza de NA como string ─────────────────────────────────────────
# ANTES: función definida en celda → no reutilizable, no testeable
# AHORA: limpiar_na_strings(df) del módulo preprocessing.py

df_clean = limpiar_na_strings(df)

print("✅ 2.1 Limpieza de NA como string completada")
print()

# Verificación
for col in ['Mto_ingreso_mensual', 'Nro_dependiente']:
    na_restantes = df_clean[col].astype(str).str.upper().str.contains('NA', na=False).sum()
    print(f"  {col}: NA como string restantes = {na_restantes}  ✓")

In [ ]:
# ── 2.2 Winsorización de outliers ──────────────────────────────────────────
# ANTES: winsorize_series(series, lower_percentile=0.05, upper_percentile=0.95)
#        llamada una por una en cada variable
# AHORA: winsorizar_columnas(df) aplica a todas las variables configuradas

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
vars_winsorizadas = ['Prct_uso_tc', 'Prct_deuda_vs_ingresos', 'Mto_ingreso_mensual']

for idx, var in enumerate(vars_winsorizadas):
    axes[0, idx].boxplot(df_clean[var].dropna(), positions=[1])
    axes[0, idx].set_title(f'{var}\nANTES', fontsize=10, fontweight='bold')
    axes[0, idx].grid(True, alpha=0.3)
    axes[0, idx].set_xticks([])

df_clean = winsorizar_columnas(df_clean)

for idx, var in enumerate(vars_winsorizadas):
    axes[1, idx].boxplot(df_clean[var].dropna(), positions=[1])
    axes[1, idx].set_title(f'{var}\nDESPUÉS', fontsize=10, fontweight='bold', color='green')
    axes[1, idx].grid(True, alpha=0.3)
    axes[1, idx].set_xticks([])

plt.suptitle('2.2 Winsorización — Antes vs Después (percentiles 5-95)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3 Imputación de valores nulos ────────────────────────────────────────
# ANTES: loop manual con fillna por variable en el notebook
# AHORA: imputar_nulos(df) con estrategias configuradas en el módulo

nulos_antes = df_clean.isnull().sum().sum()
print(f"  Valores nulos antes de imputar : {nulos_antes:,}")

df_clean = imputar_nulos(df_clean)

nulos_despues = df_clean.isnull().sum().sum()
print(f"  Valores nulos después de imputar: {nulos_despues}")

if nulos_despues == 0:
    print("\n  ✅ Dataset completamente limpio — sin valores nulos")

---
## 🧪 Parte 2 (cont.) — Feature Engineering

**Módulo:** `src/features.py`  
**Funciones:** `crear_score_retrasos()`, `crear_categorias_edad()`, `crear_categorias_dependientes()`, `estandarizar_variables()`, `seleccionar_features()`

### Variables creadas en el notebook original (ahora en funciones testeables)
| Variable | Descripción | Pesos / Bins |
|---|---|---|
| `Score_retrasos` | Score compuesto de historial de pagos | `Nro_prestao_retrasados×3 + Nro_retraso_60dias×5 + Nro_retraso_ultm3anios×2` |
| `Edad_cat` | Grupos de edad | `[<25, 25-35, 35-45, 45-55, 55-65, >65]` |
| `Deps_cat` | Grupos de dependientes | `[0, 1-2, 3-4, 5+]` |
| `*_std` | Variables estandarizadas | StandardScaler (media=0, std=1) |

In [ ]:
# ── Feature Engineering ────────────────────────────────────────────────────
# ANTES: código suelto en celdas calculando score, bins, dummies, scaler
# AHORA: funciones en features.py con pesos exactos del notebook original

df_feat = crear_score_retrasos(df_clean)
df_feat = crear_categorias_edad(df_feat)
df_feat = crear_categorias_dependientes(df_feat)
df_feat = estandarizar_variables(df_feat)

print("✅ Feature engineering completado")
print()
print("  Variables nuevas creadas:")
nuevas = [c for c in df_feat.columns if c not in df.columns]
for col in nuevas:
    print(f"    + {col}")

In [ ]:
# ── Verificación del Score_retrasos ────────────────────────────────────────
print("📊 DISTRIBUCIÓN DEL SCORE DE RETRASOS")
print(f"  Rango  : [{df_feat['Score_retrasos'].min():.0f}, {df_feat['Score_retrasos'].max():.0f}]")
print(f"  Mediana: {df_feat['Score_retrasos'].median():.1f}")
print(f"  Media  : {df_feat['Score_retrasos'].mean():.2f}")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_feat[df_feat['Default']==0]['Score_retrasos'],
             bins=30, alpha=0.6, color='#2ecc71', label='No Default', density=True)
axes[0].hist(df_feat[df_feat['Default']==1]['Score_retrasos'],
             bins=30, alpha=0.6, color='#e74c3c', label='Default', density=True)
axes[0].set_title('Score_retrasos por Default', fontweight='bold')
axes[0].set_xlabel('Score de Retrasos')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

edad_default = df_feat.groupby('Edad_cat')['Default'].mean() * 100
edad_default.plot(kind='bar', ax=axes[1], color='#3498db', alpha=0.8)
axes[1].set_title('Tasa de Default por Grupo de Edad', fontweight='bold')
axes[1].set_ylabel('Tasa de Default (%)')
axes[1].set_xlabel('Grupo de Edad')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── Selección de features con SelectKBest ──────────────────────────────────
# ANTES: código de 30 líneas con SelectKBest en celda del notebook
# AHORA: seleccionar_features(X, y, k=15) del módulo features.py

TARGET   = 'Default'
ID_COL   = 'ID'
EXCLUIR  = [TARGET, ID_COL, 'Edad_cat', 'Deps_cat']
K_FEATS  = 15

X_all = df_feat.drop(columns=[c for c in EXCLUIR if c in df_feat.columns])
X_all = X_all.select_dtypes(include=['number'])
y     = df_feat[TARGET]

top_features = seleccionar_features(X_all, y, k=K_FEATS)
X_sel = X_all[top_features]

print(f"\n📊 TOP {K_FEATS} VARIABLES SELECCIONADAS (SelectKBest - f_classif):")
print("-" * 50)
for i, feat in enumerate(top_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\n  Forma de X final: {X_sel.shape}")

---
## ⚖️ Parte 3 — División y Balanceo de Datos

**Módulo:** `src/preprocessing.py`  
**Función:** `dividir_y_balancear(X, y, test_size=0.2, tecnica='smote')`

### Técnica seleccionada: SMOTE
El notebook original comparó 4 técnicas. Para producción se usa **SMOTE** por su mejor balance entre preservación de estructura y score de balance. La función encapsula tanto el `train_test_split` estratificado como el balanceo.

In [ ]:
# ── División y balanceo ─────────────────────────────────────────────────────
# ANTES: train_test_split + SMOTE en celdas separadas, sin encapsular
# AHORA: dividir_y_balancear() hace ambas operaciones en orden correcto

X_train, X_test, y_train, y_test = dividir_y_balancear(
    X_sel, y,
    test_size=0.2,
    tecnica='smote',
    random_state=42,
)

print("✅ División y balanceo completados")
print()
print(f"  Train (antes de SMOTE): {Counter(y_train[:int(len(y_train)*0.9)])}")
print(f"  Train (después SMOTE) : {Counter(y_train)}")
print(f"  Test  (sin cambios)   : {Counter(y_test)}")
print()

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

dist_orig  = Counter(y)
dist_train = Counter(y_train)
dist_test  = Counter(y_test)

for ax, dist, titulo, color_0, color_1 in [
    (axes[0], dist_orig,  'Dataset Original',  '#e74c3c', '#c0392b'),
    (axes[1], dist_train, 'Train (post-SMOTE)', '#2ecc71', '#27ae60'),
    (axes[2], dist_test,  'Test (estratificado)', '#3498db', '#2980b9'),
]:
    total = sum(dist.values())
    bars = ax.bar(['No Default', 'Default'], [dist[0], dist[1]],
                  color=[color_0, color_1], alpha=0.8)
    for bar, cnt in zip(bars, [dist[0], dist[1]]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(dist.values())*0.02,
                f'{cnt:,}\n({cnt/total*100:.1f}%)', ha='center', fontsize=9)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Muestras')
    ax.grid(True, alpha=0.3, axis='y')
    ratio = dist[0]/dist[1] if dist[1] > 0 else 0
    ax.text(0.5, 0.95, f'Ratio: {ratio:.1f}:1', transform=ax.transAxes,
            ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('Comparación de Distribución por Conjunto', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🤖 Parte 4 — Modelado

**Módulo:** `src/train.py`  
**Funciones:** `comparar_modelos()`, `entrenar_modelo()`, `analizar_errores()`

### Algoritmos (idénticos al notebook original)
1. **Regresión Logística** — lineal, interpretable, baseline
2. **Random Forest** — ensemble robusto, importancia de variables
3. **Gradient Boosting** — alto performance, relaciones complejas

### ¿Qué cambió?
La función `entrenar_evaluar_modelo()` del notebook era una función local en celda. Ahora vive en `train.py`, tiene docstring, logging y tipos de retorno explícitos.

In [ ]:
# ── Entrenamiento de los 3 modelos ──────────────────────────────────────────
# ANTES: entrenar_evaluar_modelo() definida en celda, llamada 3 veces manualmente
# AHORA: comparar_modelos() entrena todos y devuelve resultados ordenados

print("🚀 Entrenando los 3 modelos...")
print("   (Esto puede tomar 20-40 segundos con Gradient Boosting)")
print()

resultados = comparar_modelos(X_train, y_train, X_test, y_test)

print()
print("=" * 65)
print(f"  {'MODELO':<25} {'F1':>7} {'RECALL':>8} {'PREC':>8} {'AUC':>8}")
print("-" * 65)
for nombre, res in resultados.items():
    m = res['metricas']
    auc = f"{m['roc_auc']:.4f}" if m['roc_auc'] else "  N/A "
    print(f"  {nombre:<25} {m['f1']:>7.4f} {m['recall']:>8.4f} {m['precision']:>8.4f} {auc:>8}")
print("=" * 65)

In [ ]:
# ── Visualización comparativa de métricas ──────────────────────────────────
nombres   = list(resultados.keys())
metricas_keys = ['accuracy', 'precision', 'recall', 'f1']
labels    = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colores   = ['#3498db', '#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: comparación de las 4 métricas
x = np.arange(len(metricas_keys))
width = 0.25
for i, (nombre, color) in enumerate(zip(nombres, colores)):
    m = resultados[nombre]['metricas']
    vals = [m[k] for k in metricas_keys]
    bars = axes[0].bar(x + i*width, vals, width, label=nombre, color=color, alpha=0.8)
    for bar, v in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{v:.3f}', ha='center', fontsize=7.5)

axes[0].set_title('Comparación de Métricas por Modelo', fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(labels)
axes[0].set_ylim([0, 1.1])
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Gráfico 2: ROC-AUC
auc_vals = [resultados[n]['metricas']['roc_auc'] or 0 for n in nombres]
bars2 = axes[1].bar(nombres, auc_vals, color=colores, alpha=0.8)
for bar, v in zip(bars2, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{v:.4f}', ha='center', fontweight='bold')
axes[1].set_title('ROC-AUC por Modelo', fontweight='bold')
axes[1].set_ylim([0, 1.1])
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Aleatório')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Identificar el mejor modelo ─────────────────────────────────────────────
mejor_nombre  = max(resultados, key=lambda k: resultados[k]['metricas']['f1'])
mejor_modelo  = resultados[mejor_nombre]['modelo']
mejor_metrics = resultados[mejor_nombre]['metricas']

print(f"🏆 MEJOR MODELO: {mejor_nombre}")
print(f"   F1-Score : {mejor_metrics['f1']:.4f}")
print(f"   Recall   : {mejor_metrics['recall']:.4f}  ← métrica principal (Parte 5)")
print(f"   Precision: {mejor_metrics['precision']:.4f}")
print(f"   ROC-AUC  : {mejor_metrics['roc_auc']:.4f}")

In [ ]:
# ── Importancia de variables del mejor modelo ───────────────────────────────
if hasattr(mejor_modelo, 'feature_importances_'):
    importancias = pd.DataFrame({
        'Variable':   X_sel.columns,
        'Importancia': mejor_modelo.feature_importances_
    }).sort_values('Importancia', ascending=False).head(12)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, len(importancias)))
    bars = ax.barh(range(len(importancias)), importancias['Importancia'], color=colors_imp)
    ax.set_yticks(range(len(importancias)))
    ax.set_yticklabels(importancias['Variable'])
    ax.invert_yaxis()
    ax.set_xlabel('Importancia', fontsize=11)
    ax.set_title(f'Top 12 Variables — {mejor_nombre}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

    for bar, val in zip(bars, importancias['Importancia']):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5)

    plt.tight_layout()
    plt.show()
elif hasattr(mejor_modelo, 'coef_'):
    coefs = pd.DataFrame({
        'Variable':   X_sel.columns,
        'Coeficiente': mejor_modelo.coef_[0],
        'Abs':        np.abs(mejor_modelo.coef_[0])
    }).sort_values('Abs', ascending=False).head(12)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_c = ['#e74c3c' if c > 0 else '#2ecc71' for c in coefs['Coeficiente']]
    ax.barh(range(len(coefs)), coefs['Coeficiente'], color=colors_c, alpha=0.8)
    ax.set_yticks(range(len(coefs)))
    ax.set_yticklabels(coefs['Variable'])
    ax.invert_yaxis()
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Coeficiente (positivo = mayor riesgo de default)', fontsize=10)
    ax.set_title(f'Coeficientes — {mejor_nombre}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

---
## 📊 Parte 5 — Evaluación y Justificación de Métrica

**Módulo:** `src/train.py`  
**Función:** `analizar_errores(modelo, X_test, y_test)`

### ¿Por qué RECALL es la métrica principal?

| Error | Descripción | Costo estimado |
|---|---|---|
| **Falso Negativo** | Cliente en riesgo marcado como seguro | **$10,000** (default no detectado) |
| **Falso Positivo** | Cliente seguro marcado como riesgo | **$1,000** (oportunidad perdida) |

**Ratio de costo 10:1** → minimizar Falsos Negativos es 10× más valioso que minimizar Falsos Positivos.

In [ ]:
# ── Análisis de errores del mejor modelo ───────────────────────────────────
# ANTES: código de 50+ líneas calculando TN, FP, FN, TP manualmente en celda
# AHORA: analizar_errores() devuelve un dict con todas las métricas y el reporte

errores = analizar_errores(mejor_modelo, X_test, y_test)

print(f"📊 ANÁLISIS DE ERRORES — {mejor_nombre}")
print("=" * 55)
print(f"  Verdaderos Negativos (TN): {errores['TN']:>5,}  clientes buenos ✓ detectados")
print(f"  Falsos Positivos     (FP): {errores['FP']:>5,}  clientes buenos marcados como riesgo")
print(f"  Falsos Negativos     (FN): {errores['FN']:>5,}  defaults NO detectados  ⚠️")
print(f"  Verdaderos Positivos (TP): {errores['TP']:>5,}  defaults detectados ✓")
print()
print(f"  Tasa de Falsos Positivos : {errores['fpr']:.2%}")
print(f"  Tasa de Falsos Negativos : {errores['fnr']:.2%}")
print(f"  Recall (Sensibilidad)    : {errores['recall']:.2%}")
print()
costo_fn = errores['FN'] * 10000
costo_fp = errores['FP'] * 1000
print(f"  💰 Impacto financiero estimado (test set):")
print(f"     Pérdidas por FN ({errores['FN']} × $10,000) : ${costo_fn:>8,.0f}")
print(f"     Pérdidas por FP ({errores['FP']} ×  $1,000) : ${costo_fp:>8,.0f}")
print(f"     Costo total estimado               : ${costo_fn + costo_fp:>8,.0f}")

In [ ]:
# ── Matriz de confusión ─────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = mejor_modelo.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No Default', 'Default']
)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusión\n{mejor_nombre}', fontweight='bold')

# Análisis de umbrales
try:
    from sklearn.metrics import roc_curve, auc
    y_proba = mejor_modelo.predict_proba(X_test)[:, 1]
    fpr_arr, tpr_arr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr_arr, tpr_arr)

    axes[1].plot(fpr_arr, tpr_arr, color='#e74c3c', lw=2,
                 label=f'ROC-AUC = {roc_auc_val:.4f}')
    axes[1].plot([0,1],[0,1], 'k--', alpha=0.4, label='Aleatorio')
    axes[1].fill_between(fpr_arr, tpr_arr, alpha=0.1, color='#e74c3c')
    axes[1].set_xlabel('Tasa de Falsos Positivos')
    axes[1].set_ylabel('Tasa de Verdaderos Positivos (Recall)')
    axes[1].set_title('Curva ROC', fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
except Exception:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'ROC no disponible\npara este modelo',
                 ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ── Reporte de clasificación completo ──────────────────────────────────────
print("📋 REPORTE DE CLASIFICACIÓN DETALLADO")
print("=" * 55)
print(errores['reporte'])

---
## 🚀 Parte 6 — Pipeline completo en una celda

Esta celda replica exactamente lo que hace `python src/main.py` pero dentro del notebook.  
Es el resumen final de toda la refactorización: **12 líneas de negocio reales** en lugar de 900 líneas de notebook.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║           PIPELINE COMPLETO — BANCO WIESSE                      ║
# ║   Equivalente a ejecutar: python src/main.py                    ║
# ╚══════════════════════════════════════════════════════════════════╝

print("🏦 PIPELINE COMPLETO — PREDICCIÓN DEFAULT CREDITICIO BANCO WIESSE")
print("=" * 65)

# 1. Carga
df_pipeline = cargar_datos(RUTA_DATOS)

# 2. Limpieza
df_pipeline = limpiar_na_strings(df_pipeline)
df_pipeline = winsorizar_columnas(df_pipeline)
df_pipeline = imputar_nulos(df_pipeline)

# 3. Features
df_pipeline = crear_score_retrasos(df_pipeline)
df_pipeline = crear_categorias_edad(df_pipeline)
df_pipeline = crear_categorias_dependientes(df_pipeline)
df_pipeline = estandarizar_variables(df_pipeline)

# 4. Selección y split
X_p = df_pipeline.drop(columns=[c for c in EXCLUIR if c in df_pipeline.columns])
X_p = X_p.select_dtypes(include=['number'])
y_p = df_pipeline[TARGET]
top_p = seleccionar_features(X_p, y_p, k=K_FEATS)
X_train_p, X_test_p, y_train_p, y_test_p = dividir_y_balancear(X_p[top_p], y_p)

# 5. Modelado
res_p = comparar_modelos(X_train_p, y_train_p, X_test_p, y_test_p)

# 6. Evaluación
mejor_p = max(res_p, key=lambda k: res_p[k]['metricas']['f1'])
err_p   = analizar_errores(res_p[mejor_p]['modelo'], X_test_p, y_test_p)

print()
print("═" * 65)
print("  RESULTADOS FINALES")
print("═" * 65)
for nombre, r in res_p.items():
    m = r['metricas']
    print(f"  {nombre:<25}  F1={m['f1']:.4f}  Recall={m['recall']:.4f}  AUC={m['roc_auc'] or 'N/A'}")
print("─" * 65)
print(f"  🏆 Mejor modelo : {mejor_p}")
print(f"  🎯 Recall       : {err_p['recall']:.2%}  (detecta {err_p['recall']:.0%} de los defaults reales)")
print(f"  ⚠️  FN evitados  : {err_p['TP']} de {err_p['TP']+err_p['FN']} defaults detectados")
print(f"  💰 Pérdidas FN  : ${err_p['FN'] * 10000:,.0f}  |  Pérdidas FP: ${err_p['FP'] * 1000:,.0f}")
print("═" * 65)
print("  ✅ Pipeline completado")

---
## 🧪 Tests unitarios — Verificación desde el notebook

Los tests viven en `tests/` y se corren con `pytest tests/ -v`.  
Esta celda los ejecuta directamente desde el notebook para verificar que todo funciona.

In [ ]:
# ── Ejecutar tests desde el notebook ───────────────────────────────────────
import subprocess

result = subprocess.run(
    ['pytest', '../tests/', '-v', '--tb=short', '--no-header'],
    capture_output=True,
    text=True,
    cwd=os.path.join(os.getcwd(), '..', 'src')
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

---
## 📋 Resumen de la refactorización

### ¿Qué cambió entre el notebook original y este?

| Aspecto | Notebook Original | Proyecto Modular |
|---|---|---|
| **Líneas de código** | ~950 líneas en celdas | ~60 líneas en notebook + módulos en `src/` |
| **Funciones** | Definidas en celdas sueltas | En módulos con docstrings y tipos |
| **Reutilización** | No — solo en este notebook | Sí — cualquier script puede importarlas |
| **Testing** | No hay | 9 tests con pytest |
| **Logging** | `print()` | `logging` con timestamp y nivel |
| **Ejecución en producción** | Imposible | `python src/main.py` |
| **Manejo de errores** | No hay | `FileNotFoundError`, `ValueError` explícitos |
| **Reproducibilidad** | Solo si las celdas se corren en orden | Garantizada — `main.py` lo hace automático |

### Lo que NO cambió
- Los **algoritmos** son exactamente los mismos
- Los **pesos** del `Score_retrasos` (3, 5, 2) son idénticos
- Los **bins** de `Edad_cat` y `Deps_cat` son iguales
- La **técnica de balanceo** (SMOTE) es la misma
- Los **parámetros** de los modelos son los del notebook original

---

### 🚀 Próximos pasos — Tema 3
En el siguiente tema agregaremos **MLflow** para registrar automáticamente  
cada ejecución de `python src/main.py`:
- Parámetros: `k_features`, `test_size`, `tecnica_balanceo`, hiperparámetros
- Métricas: `f1`, `recall`, `roc_auc`, `FN`, `FP`
- Artefactos: modelo serializado, gráficos de importancia

```python
import mlflow
with mlflow.start_run():
    mlflow.log_params({'k_features': K_FEATS, 'tecnica': 'smote'})
    mlflow.log_metrics({'recall': errores['recall'], 'f1': mejor_metrics['f1']})
    mlflow.sklearn.log_model(mejor_modelo, 'modelo_crediticio')
```